# SafeScan Inference Server
**FastAPI + ngrok tunnel for Flutter app**

### Setup Order
1. Connect GPU first (Runtime → Change runtime type → T4)
2. Run Cell 1 → Restart runtime
3. Run Cell 2 → symlink fix
4. Run Cell 3 → load model
5. Run Cell 4 → test locally
6. Run Cell 5 → start server
7. Copy ngrok URL → paste into Flutter safescan_service.dart

## Cell 1 — Install Dependencies
Run once, then restart runtime.

In [2]:
import sys

# Core ML stack
!{sys.executable} -m pip install -q \
    transformers \
    peft \
    accelerate \
    datasets \
    scipy einops sentencepiece

# bitsandbytes
!{sys.executable} -m pip install -q bitsandbytes==0.46.1

# Server stack
!{sys.executable} -m pip install -q \
    fastapi \
    uvicorn \
    pyngrok \
    nest-asyncio \
    httpx

print('✅ All packages installed.')
print('⚠️  Restart runtime now, then run Cell 2 onwards.')

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 362, in run
    resolver = self.make_resolver(
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 177, in make_resolver
    return pip._internal.resolution.resolvelib.resolver.Resolver(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 58, in __init__
    self.factory = Factory(
                  

## Cell 2 — Fix CUDA 13 + Verify GPU

In [1]:
import os, torch

# Symlink bitsandbytes for CUDA 13
src = '/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda129.so'
dst = '/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda130.so'
if os.path.exists(src) and not os.path.exists(dst):
    os.symlink(src, dst)
    print('✅ CUDA 13 symlink created')
else:
    print('✅ Symlink already exists or not needed')

# GPU check
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1024**3
    print(f'✅ GPU:     {props.name}')
    print(f'✅ VRAM:    {vram:.1f} GB')
    print(f'✅ CUDA:    {torch.version.cuda}')
    print(f'✅ PyTorch: {torch.__version__}')
else:
    print('❌ No GPU detected — connect T4 first')

✅ CUDA 13 symlink created
✅ GPU:     Tesla T4
✅ VRAM:    14.6 GB
✅ CUDA:    12.8
✅ PyTorch: 2.11.0+cu128


## Cell 3 — Define Predict Function + Local Test *

In [6]:
import re
import json
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ==========================================================
# ⚡ LOAD MODEL
# ==========================================================
HUGGINGFACE_REPO = 'MuhammadSanan99989/safescan-phi3-mini-intent-gemini'

print("🔄 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(HUGGINGFACE_REPO, trust_remote_code=False)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("🛠️ Sanitizing config...")
config = AutoConfig.from_pretrained(HUGGINGFACE_REPO)
if hasattr(config, "auto_map"):
    delattr(config, "auto_map")
if hasattr(config, "rope_scaling") and config.rope_scaling is not None:
    if "rope_type" not in config.rope_scaling:
        config.rope_scaling["rope_type"] = config.rope_scaling.get("type", "default")

print("⚡ Loading quantized model...")
q_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)
model = AutoModelForCausalLM.from_pretrained(
    HUGGINGFACE_REPO,
    config=config,
    quantization_config=q_config,
    device_map="auto",
    trust_remote_code=False,
    attn_implementation='eager'
)
model.eval()
print("✅ Model ready\n")

# ==========================================================
# 🎯 PREDICT FUNCTION
# ==========================================================
STOP_ID = tokenizer.convert_tokens_to_ids("<|end|>")
if STOP_ID is None or STOP_ID == tokenizer.unk_token_id:
    STOP_ID = tokenizer.eos_token_id

def predict(user_text: str) -> dict:
    prompt = f'<|user|>\n<|user|>\n{user_text}<|end|>\n<|assistant|><|end|>\n<|assistant|>'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            eos_token_id=STOP_ID,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    generated_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    decoded = re.sub(r'^```json\s*|```$', '', decoded, flags=re.MULTILINE).strip()

    if '}' in decoded:
        decoded = decoded.split('}')[0] + '}'

    try:
        return json.loads(decoded)
    except json.JSONDecodeError:
        intent_match = re.search(r'"intent":\s*"([^"]+)"', decoded)
        if intent_match:
            return {'intent': intent_match.group(1).strip(), 'raw': decoded}
        return {'intent': 'parse_error', 'raw': decoded}


# ==========================================================
# 💬 CHAT LOOP
# ==========================================================
print("💬 SafeScan Chat — type 'exit' to quit\n")
print("=" * 50)

while True:
    user_input = input("\n You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ['exit', 'quit', 'q']:
        print("👋 Exiting.")
        break

    result = predict(user_input)
    print(f"\n SafeScan: {json.dumps(result, indent=2)}")
    print("-" * 50)

🔄 Loading tokenizer...
🛠️ Sanitizing config...
⚡ Loading quantized model...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model ready

💬 SafeScan Chat — type 'exit' to quit


🧑 You: hey

🤖 SafeScan: {
  "intent": "general_chat",
  "module": null,
  "action": "show_text_response",
  "parameters": null,
  "response": "Hey! SafeScan AI here. What security task can I help you with today?"
}
--------------------------------------------------

🧑 You: is my wifi safe

🤖 SafeScan: {
  "intent": "wifi_check",
  "raw": "{\"intent\": \"wifi_check\", \"module\": \"WifiSecurityScan\", \"action\": \"navigate_to_wifi_security_scan\", \"parameters\": {\"network\": null}"
}
--------------------------------------------------

🧑 You: wifi

🤖 SafeScan: {
  "intent": "wifi_check",
  "raw": "{\"intent\": \"wifi_check\", \"module\": \"WifiSecurityScan\", \"action\": \"navigate_to_wifi_security_scan\", \"parameters\": {\"network\": null}"
}
--------------------------------------------------

🧑 You: exit
👋 Exiting.


## Cell 4 — Start FastAPI Server + ngrok Tunnel
Replace `YOUR_NGROK_TOKEN` with your token from https://ngrok.com

The printed URL goes into Flutter's `safescan_service.dart` as the base URL.

In [ ]:
import asyncio
import nest_asyncio
import uvicorn
import re
import json
import torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForCausalLM

# Patch the event loop to allow nested loops inside Jupyter/Colab
nest_asyncio.apply()

# ==========================================================
# BULLETPROOF VARIABLE INITIALIZATION LAYER
# ==========================================================
HUGGINGFACE_REPO = 'MuhammadSanan99989/safescan-phi3-mini-intent-gemini'

# Check if the variables exist in the global scope; if not, load them right here
if 'eval_tokenizer' not in globals() or 'eval_model' not in globals():
    print("🔄 Session memory empty. Automatically loading tokenizer and model weights...")

    eval_tokenizer = AutoTokenizer.from_pretrained(HUGGINGFACE_REPO, trust_remote_code=False)

    # Check if a GPU is available to prevent slow CPU generation bottlenecks
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"📡 Loading model onto device target: {device}")

    eval_model = AutoModelForCausalLM.from_pretrained(
        HUGGINGFACE_REPO,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto"
    )
    eval_model.eval()
    print("✅ Model components loaded successfully into memory.")
else:
    print("✅ Active model weights found in memory scope. Skipping initialization step.")

# Resolve the stop token structure
STOP_TOKEN_ID = eval_tokenizer.convert_tokens_to_ids("<|end|>")
if STOP_TOKEN_ID is None or STOP_TOKEN_ID == eval_tokenizer.unk_token_id:
    STOP_TOKEN_ID = eval_tokenizer.eos_token_id

# ==========================================================
# 🔮 INFERENCE PREDICT FUNCTION
# ==========================================================
def predict(user_text: str) -> dict:
    """Run deterministic intent routing with aggressive parsing fallback layers."""
    prompt = f'<|user|>\n<|user|>\n{user_text}<|end|>\n<|assistant|><|end|>\n<|assistant|>'
    inputs = eval_tokenizer(prompt, return_tensors='pt').to(eval_model.device)

    with torch.no_grad():
        output = eval_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            eos_token_id=STOP_TOKEN_ID,
            pad_token_id=eval_tokenizer.eos_token_id,
            use_cache=True,
        )

    generated_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = eval_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    decoded = re.sub(r'^```json\s*|```$', '', decoded, flags=re.MULTILINE).strip()

    if '}' in decoded:
        cleaned_json = decoded.split('}')[0] + '}'
    else:
        cleaned_json = decoded

    try:
        return json.loads(cleaned_json)
    except json.JSONDecodeError:
        intent_match = re.search(r'"intent":\s*"([^"]+)"', decoded)
        if intent_match:
            return {
                'intent': intent_match.group(1).strip(),
                'module': 'SalvagedPayload',
                'action': 'regex_extraction_fallback',
                'parameters': {},
                'raw_salvage': decoded
            }
        return {
            'intent': 'parse_error',
            'raw_output': decoded
        }

# ==========================================================
# 📡 FASTAPI / UVICORN PIPELINE
# ==========================================================
app = FastAPI(title='SafeScan Inference API Router', description='Production API Pipeline')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

class PredictRequest(BaseModel):
    message: str

@app.get('/health')
def health():
    return {'status': 'ok', 'model': 'safescan-phi3-mini-merged'}

@app.post('/predict')
def predict_endpoint(req: PredictRequest):
    if not req.message.strip():
        raise HTTPException(status_code=400, detail='Message string cannot be empty.')

    result = predict(req.message)
    return result

# ── Launch Public Ngrok Tunnel ───────────────────────────────────────────────
NGROK_TOKEN = '3EV1QEBC7sqb4CbkmoMcfsY7rla_6yxALmzXQ1J27NcL97iGg'
ngrok.set_auth_token(NGROK_TOKEN)

# Drop any zombie ngrok connections
ngrok.kill()

public_url = ngrok.connect(8000)
clean_url = str(public_url).split('"')[1] if '"' in str(public_url) else str(public_url).replace('NgrokTunnel: ', '').split(' -> ')[0]

print('=' * 65)
print(f'🚀 SafeScan Production Server is LIVE')
print(f'🌐 Base Tunnel Domain : {clean_url}')
print(f'📡 Flutter App Target : {clean_url}/predict')
print('=' * 65)
print('Streaming real-time request logs below. Press the STOP button to terminate.')
print('=' * 65)

# ── Real-Time Streaming Event Loop ───────────────────────────────────────────
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())

🔄 Session memory empty. Automatically loading tokenizer and model weights...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.62M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

✅ Model components loaded successfully into memory.


INFO:     Started server process [1034]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 SafeScan Production Server is LIVE
🌐 Base Tunnel Domain : https://voter-cardiac-roundish.ngrok-free.dev
📡 Flutter App Target : https://voter-cardiac-roundish.ngrok-free.dev/predict
Streaming real-time request logs below. Press the STOP button to terminate.
